In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import sys
repo_path = Path("../../anomolous_ts")
if str(repo_path) not in sys.path:
    sys.path.append(str(repo_path))

from anomolous_ts.anomolize import TimeseriesGenerator
from anomolous_ts.isoforest import TimeSeriesIsolationForest, IsolationForestConfig
from anomolous_ts.feature_extractors import TimeSeriesPreprocessor
from anomolous_ts.visualizer import TimeSeriesVisualizer


In [2]:

# Example 1: Basic Usage with Univariate Time Series

# Generate sample time series data with known anomalies
generator = TimeseriesGenerator(start_date='2024-01-01', periods=500)
data, true_anomalies = generator.generate_correlated_series(
    n_series=1,  # Single time series
    seasonal_periods=[24, 168],  # Daily and weekly seasonality
    trend_slopes=[0.05],
    noise_levels=[0.1],
    num_anomalies=10
)


# Create configuration
config = IsolationForestConfig(
    window_size=24,  # 24-hour window
    n_estimators=100,
    contamination='auto',
    seasonal_period=24,
    feature_settings={
        'statistical': True,
        'spectral': False,
        'wavelet': False
    }
)



# Initialize preprocessor
preprocessor = TimeSeriesPreprocessor()

# Initialize detector with config
detector = TimeSeriesIsolationForest(config, preprocessor)

# Fit and predict
detector.fit(data['series_1'])
result = detector.predict(data['series_1'])

In [3]:
result 

In [4]:




# Extract anomalies and plot
anomalies = pd.Series(
    [score.is_anomaly for score in result.scores],
    index=data.index
)
confidence_scores = pd.Series(
    [score.confidence for score in result.scores],
    index=data.index
)

# Visualize results
visualizer = TimeSeriesVisualizer()
visualizer.plot_anomalies(
    data['series_1'],
    anomalies,
    confidence_scores,
    title='Basic Anomaly Detection Example'
)

# Print detection results
print("\nDetection Results:")
print(f"Number of anomalies detected: {anomalies.sum()}")
print(f"Average confidence score: {confidence_scores.mean():.3f}")

# Compare with true anomalies
true_dates = true_anomalies['series_1']
detected_dates = data.index[anomalies]
print("\nTrue anomaly dates:", true_dates.tolist())
print("Detected anomaly dates:", detected_dates.tolist())


AttributeError: 'NoneType' object has no attribute 'scores'

In [ ]:

# Example 2: Multivariate Time Series with Preprocessing
def multivariate_example():
    # Generate multivariate time series
    generator = TimeseriesGenerator(start_date='2024-01-01', periods=1000)
    correlation_matrix = np.array([
        [1.0, 0.7, -0.3],
        [0.7, 1.0, -0.5],
        [-0.3, -0.5, 1.0]
    ])
    
    data, true_anomalies = generator.generate_correlated_series(
        n_series=3,
        correlation_matrix=correlation_matrix,
        seasonal_periods=[24, 168],  # Daily and weekly patterns
        trend_slopes=[0.1, 0.05, -0.08],
        noise_levels=[0.1, 0.15, 0.12],
        num_anomalies=15
    )
    
    # Initialize preprocessor with advanced settings
    preprocessor = TimeSeriesPreprocessor(
        imputation_method='hybrid',
        scaling_method='robust'
    )
    
    # Initialize detector with preprocessor
    detector = AdvancedTimeSeriesIsolationForest(
        window_size=24,
        n_estimators=200,
        contamination=0.02,
        seasonal_period=24,
        preprocessor=preprocessor
    )
    
    # Detect anomalies across all series
    anomalies, confidence_scores = detector.detect_and_visualize(
        data,
        title='Multivariate Anomaly Detection'
    )
    
    # Print results for each series
    for column in data.columns:
        print(f"\nResults for {column}:")
        print(f"Number of anomalies: {anomalies[column].sum()}")
        print(f"Average confidence: {confidence_scores[column].mean():.3f}")
        print("True anomaly dates:", true_anomalies[column].tolist())
        print("Detected dates:", data.index[anomalies[column]].tolist())


In [ ]:

# Example 3: Streaming Data Processing
def streaming_example():
    # Create a data generator
    generator = TimeseriesGenerator(start_date='2024-01-01', periods=100)
    
    # Create a streaming detector
    detector = AdvancedTimeSeriesIsolationForest(
        window_size=10,
        n_estimators=50,
        contamination=0.1
    )
    
    # Simulate streaming data
    def data_stream():
        while True:
            data, _ = generator.generate_correlated_series(
                n_series=1,
                seasonal_periods=[24],
                num_anomalies=2
            )
            for value in data['series_1']:
                yield pd.Series([value])
    
    # Process stream with custom callback
    def callback(result):
        if any(score.is_anomaly for score in result.scores):
            print(f"Anomaly detected at {result.detection_time}")
            print(f"Confidence scores: {[score.confidence for score in result.scores]}")
    
    # Process stream for a few iterations
    stream = data_stream()
    for _ in range(5):  # Process 5 chunks
        chunk_data = pd.concat([next(stream) for _ in range(20)])  # 20 points per chunk
        anomalies = detector.fit_predict(chunk_data)
        callback(anomalies)

if __name__ == "__main__":
    print("Running Basic Example...")
    basic_example()
    
    print("\nRunning Multivariate Example...")
    multivariate_example()
    
    print("\nRunning Streaming Example...")
    streaming_example()